In [ ]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
if torch.backends.mps.is_available():
    print("Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).")
    mps_enabled = True
    torch.set_default_dtype(torch.float32)
    print("set default to float32")

In [ ]:
import keras
import tensorflow as tf

In [ ]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [ ]:
keras.backend.backend()

In [ ]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]
input_size = config["inputSize"]
output_size = config["outputSize"]
seed = config["seed"]

In [ ]:
keras.utils.set_random_seed(seed)

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
# Dataset initialization

from utils.data_loader import get_ml_cup_data, split_dataloader, cv_fold_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    match scaler_type:
        case "Standard":
            return StandardScaler()
        case "MinMax":
            return MinMaxScaler()
        case "Robust":
            return RobustScaler()
        case "MaxAbsScaler":
            return MaxAbsScaler()
        case _:
            return None

train_loader, test_loader = get_ml_cup_data(
    batch_size, 
    scaler=_scaler(),
    mps=mps_enabled
    )

In [ ]:
train_loader.dataset.X.shape, train_loader.dataset.y.shape

In [ ]:
test_loader.dataset.X.shape, test_loader.dataset.y.shape

In [ ]:
test_loader.dataset.y.shape[1]

In [ ]:
import numpy as np

y_mean = train_loader.dataset.y.mean(axis=0)        # (4,)

y_pred_baseline = np.tile(y_mean, (len(train_loader.dataset.y), 1))

mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)

mee_baseline = mee_errors.mean()
mse_baseline = mse_errors.mean()

print("Baseline MEE:", mee_baseline)
print("Baseline MSE:", mse_baseline)

In [ ]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee", dtype=torch.float32)

In [ ]:
train_dataset = train_loader.dataset
test_dataset = test_loader.dataset

## Randomized Search

In [ ]:
from scipy.stats import loguniform

param_distributions = {
    "reg__model__learning_rate": loguniform(1e-3, 1e-2),
    "reg__model__lambda_1": loguniform(3e-3, 1e-1),
    "reg__model__lambda_2": loguniform(3e-3, 1e-1),
    "reg__model__activation_1": ["relu", "gelu", "leaky_relu"],
    "reg__model__activation_2": ["relu", "gelu", "leaky_relu"],
    "reg__model__dropout_1": loguniform(0.2, 0.5),
    "reg__model__dropout_2": loguniform(0.2, 0.5),
    "reg__model__pca_input_size": [1, 2],
    "reg__pca__n_components": [1, 2],
    "reg__model__seed": [seed],
    "reg__model__output_size": [train_loader.dataset.y.shape[1]],
}

In [ ]:
from executors import RandomizedSearchRegressionExecutor

# using default param_distribution
rs_regression_executor = RandomizedSearchRegressionExecutor(
    train_loader=train_loader,
    units=[(2,2)],
    n_iter=1,
    epochs=1,
    use_PCA=True,
    loss="mean_squared_error",
    baseline=mse_baseline,
    scoring="neg_mean_squared_error",
    seed=seed,
    save_path="keras/models/rs/test",
    verbose=0,
    n_jobs=4
)

In [ ]:
rs_regression_executor.execute()

## Optuna

In [ ]:
import optuna
from executors import OptunaRegressorExecutor

optuna_executor = OptunaRegressorExecutor(
    train_loader=train_loader,
    units=[(12,12)],
    use_pca=True,
    pca_input_size=2,
    n_trials=100,
    epochs=1000,
    seed=seed,
    batch_size=80,
    n_splits=5,
    sampler=optuna.samplers.TPESampler(seed=seed, constant_liar=True, multivariate=True),
    optuna_base_path="keras/models/optuna/regression27-12",
    verbose=0,
    baseline=mse_baseline,
    n_jobs=4
)

In [ ]:
optuna_executor.execute()

In [ ]:
import utils.optuna as uoptuna

study = uoptuna.import_csv("keras/models/optuna/regression27-12/12x12/optuna_results.csv")

In [ ]:
from optuna.visualization import \
    plot_optimization_history, plot_param_importances, plot_parallel_coordinate, plot_contour

In [ ]:
plot_optimization_history(study)

In [ ]:
study.best_params

In [ ]:
plot_param_importances(study)

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_contour(study, params=['dropout_2', 'learning_rate'])

In [ ]:
from utils.plot import plot_optuna_vs_random

plot_optuna_vs_random(
    optuna_csv_path="keras/models/optuna/regression27-12/12x12/optuna_results.csv",
    rs_csv_path="keras/models/rs/regression27-12/12x12/cv_results_df.csv"
                      )